In [ ]:
import sys

# ============================================================
# ADD PATHS SO PYTHON CAN FIND EACH MODEL FILE
# Each file contains a create_model() function
# ============================================================

sys.path.append(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\04. ten models comparison\RF.py")
import RF

sys.path.append(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\04. ten models comparison\SVM.py")
import SVM

sys.path.append(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\04. ten models comparison\NN.py")
import NN

sys.path.append(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\04. ten models comparison\KNN.py")
import KNN

sys.path.append(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\04. ten models comparison\KM.py")
import KM

sys.path.append(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\04. ten models comparison\NB.py")
import NB

sys.path.append(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\04. ten models comparison\LR.py")
import LR

sys.path.append(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\04. ten models comparison\LD.py")
import LD

sys.path.append(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\04. ten models comparison\GBC.py")
import GBC

sys.path.append(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\04. ten models comparison\DT.py")
import DT


# ============================================================
# MODEL REGISTRY (MAP NUMBER → MODEL MODULE)
# This allows selecting models using numbers (1–10)
# ============================================================

model_map = {1: RF,2: SVM,3: NN,4: KNN,5: KM,6: NB,7: LR,8: LD,9: GBC,10: DT}

# ============================================================
# MODEL NAME MAP (NUMBER → SHORT NAME)
# Used for display/output purposes
# ============================================================

model_names = {1: "RF",2: "SVM",3: "NN",4: "KNN",5: "KM",6: "NB",7: "LR",8: "LD",9: "GBC",10: "DT"}

# ============================================================
# USER INPUT: SELECT WHICH MODELS TO RUN
# Example input format: 1,3,5
# ============================================================

selected_numbers = input("Enter model numbers to run")

# Convert input string into a list of integers
selected_numbers = [int(x.strip()) for x in selected_numbers.split(",")]

In [ ]:
# =========================
# 1. Import libraries
# =========================
import pandas as pd
import numpy as np
from itertools import combinations

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


# ============================================================
# 2. SELECT MODELS BASED ON USER INPUT (FROM CONTROLLER)
# ============================================================

selected_models = {num: model_map[num] for num in selected_numbers if num in model_map}

# Create a tag for saving results file name
model_tag = "_".join(model_names[num] for num in selected_numbers if num in model_names)


# ============================================================
# 3. LOAD DATASETS
# ============================================================

main_data_path = r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\02. data\Full data.csv"
selected_features_path = r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\03. features selection\features.csv
df = pd.read_csv(main_data_path)
selected_features_df = pd.read_csv(selected_features_path)

# Clean column names
df.columns = df.columns.str.strip()
selected_features_df["Feature"] = selected_features_df["Feature"].str.strip()


# ============================================================
# 4. MULTI-LABEL OUTPUT CREATION
# Convert "Pictogram(s)" column into multiple binary output columns
# ============================================================

df["Pictogram(s)"] = df["Pictogram(s)"].astype(str).str.split(",")
all_outputs = set([item.strip() for sublist in df["Pictogram(s)"] for item in sublist])
for output in all_outputs:
    df[output] = df["Pictogram(s)"].apply(lambda x: int(output in [i.strip() for i in x]))

outputs = [o for o in all_outputs if o in df.columns]


# ============================================================
# 5. FEATURE SELECTION
# ============================================================

top_features = selected_features_df["Feature"].tolist()
top_features = [f for f in top_features if f in df.columns]


# ============================================================
# 6. STORE RESULTS
# ============================================================

results_dict = {}


# ============================================================
# 7. MODEL TRAINING LOOP
# Loop through each output (label)
# Loop through all feature combinations
# Train all selected models
# ============================================================

for output in outputs:
    if output not in df.columns:
        print(f"Skipping '{output}' as it is not in the dataset")
        continue

    y = df[output]

    # Ensure features still exist in dataset
    top_features = [f for f in top_features if f in df.columns]

    # Try all feature combinations (1 feature → full set)
    for n in range(1, len(top_features) + 1):
        for feature_comb in combinations(top_features, n)
            feature_list = list(feature_comb)
            feature_label = "_".join([f.replace(" ", "_") for f in feature_list])
            print(f"Training models for output '{output}' with features: {feature_list}")

            # Select input features
            X = df[feature_list]

            # Split dataset into train/test sets
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

            # ====================================================
            # DATA PREPROCESSING
            # ====================================================

            # Handle missing values
            imputer = SimpleImputer(strategy="mean")
            X_train = imputer.fit_transform(X_train)
            X_test = imputer.transform(X_test)

            # Standardize features
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)

            # ====================================================
            # MODEL TRAINING + EVALUATION
            # ====================================================

            for number, module in selected_models.items():

                # Create model instance
                model = module.create_model()

                # Train model
                model.fit(X_train, y_train)

                # Predict
                predictions = model.predict(X_test)

                # Evaluate performance
                acc = accuracy_score(y_test, predictions)
                precision = precision_score(y_test, predictions, zero_division=0)
                recall = recall_score(y_test, predictions, zero_division=0)
                f1 = f1_score(y_test, predictions, zero_division=0)

                # Store results
                key = (output, feature_label)
                results_dict[key] = {"Output": output,"Features Combination": feature_label,"Accuracy": acc,"f1score": f1,"presicion": precision,"recall": recall}


# ============================================================
# SAVE RESULTS TO CSV
# ============================================================

results_df = pd.DataFrame(results_dict.values())
results_df.to_csv(f"Results_{model_tag}.csv", index=False)
print("done!")